<a href="https://colab.research.google.com/github/asmaslenikova/maslenikova-compling/blob/main/rag_langchain_arxiv_%D0%9C%D0%B0%D1%81%D0%BB%D0%B5%D0%BD%D0%B8%D0%BA%D0%BE%D0%B2%D0%B0_%D0%9F%D0%B8%D0%B2%D0%BD%D0%B5%D0%B2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Тема**: RAG (Retrieval-Augmented Generation) с фреймворком LangChain

**Выполнили**: Масленикова Александра, Пивнев Иван, БФЛ 222


In [1]:
!pip install -q langchain langchain-core langchain-community langchain-groq \
    langchain-huggingface langchain-text-splitters \
    faiss-cpu sentence-transformers pypdf datasets tiktoken kagglehub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


## Загрузить набор текстовых документов

In [2]:
from google.colab import userdata
import os

os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "arxiv-metadata-oai-snapshot.json"

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "Cornell-University/arxiv",
    file_path,
    pandas_kwargs={"lines": True, "nrows": 200000},
)
df["id"] = df["id"].apply(lambda x: str(x).zfill(9))

df.head(5)

/tmp/ipykernel_3276/4173084327.py:11: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


100%|██████████| 4.82G/4.82G [02:13<00:00, 38.6MB/s]


,id,submitter,authors,title,comments,journal-ref,doi,report-no,categories,license,abstract,versions,update_date,authors_parsed
0,0704.0001,Pavel Nadolsky,"C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-...",Calculation of prompt diphoton production cros...,"37 pages, 15 figures; published version","Phys.Rev.D76:013009,2007",10.1103/PhysRevD.76.013009,ANL-HEP-PR-07-12,hep-ph,None,A fully differential calculation in perturba...,"[{'version': 'v1', 'created': 'Mon, 2 Apr 2007...",2008-11-26,"[[Balázs, C., ], [Berger, E. L., ], [Nadolsky,..."
1,0704.0002,Louis Theran,Ileana Streinu and Louis Theran,Sparsity-certifying Graph Decompositions,To appear in Graphs and Combinatorics,None,None,None,math.CO cs.CG,http://arxiv.org/licenses/nonexclusive-distrib...,"We describe a new algorithm, the $(k,\ell)$-...","[{'version': 'v1', 'created': 'Sat, 31 Mar 200...",2008-12-13,"[[Streinu, Ileana, ], [Theran, Louis, ]]"
2,0704.0003,Hongjun Pan,Hongjun Pan,The evolution of the Earth-Moon system based o...,"23 pages, 3 figures",None,None,None,physics.gen-ph,None,The evolution of Earth-Moon system is descri...,"[{'version': 'v1', 'created': 'Sun, 1 Apr 2007...",2008-01-13,"[[Pan, Hongjun, ]]"
3,0704.0004,David Callan,David Callan,A determinant of Stirling cycle numbers counts...,11 pages,None,None,None,math.CO,None,We show that a determinant of Stirling cycle...,"[{'version': 'v1', 'created': 'Sat, 31 Mar 200...",2007-05-23,"[[Callan, David, ]]"
4,0704.0005,Alberto Torchinsky,Wael Abu-Shammala and Alberto Torchinsky,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,None,"Illinois J. Math. 52 (2008) no.2, 681-689",None,None,math.CA math.FA,None,In this paper we show how to compute the $\L...,"[{'version': 'v1', 'created': 'Mon, 2 Apr 2007...",2013-10-15,"[[Abu-Shammala, Wael, ], [Torchinsky, Alberto, ]]"


In [3]:
filtered = df[df["categories"].str.contains("cs.LG", na=False)].head(150)

print(f"Отобрано статей: {len(filtered)}")
print("Пример id:", filtered["id"].iloc[0])

ids = filtered["id"].tolist()

Отобрано статей: 150
Пример id: 0704.0671


In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re

In [5]:
all_documents = []
failed_ids = []

ids_to_load = ids[:150]

for arxiv_id in ids_to_load:
    url = f"https://arxiv.org/pdf/{arxiv_id}"
    try:
        loader = PyPDFLoader(url)
        docs = loader.load()

        # Добавляем метаданные: id, title, категория
        row = filtered[filtered["id"] == arxiv_id].iloc[0]
        for doc in docs:
            doc.metadata["arxiv_id"] = arxiv_id
            doc.metadata["title"] = row.get("title", "").replace("\n", " ")
            doc.metadata["categories"] = row.get("categories", "")
            doc.metadata["source"] = url

        all_documents.extend(docs)
    except Exception as e:
        failed_ids.append(arxiv_id)
        print(f"  [!] Не удалось загрузить {arxiv_id}: {e}")

print(f"\nУспешно загружено статей: {len(ids_to_load) - len(failed_ids)}")
print(f"Ошибок загрузки: {len(failed_ids)}")
print(f"Всего страниц (Document объектов): {len(all_documents)}")


  [!] Не удалось загрузить 00704.102: Check the url of your file; returned status code 404
  [!] Не удалось загрузить 0704.1409: Check the url of your file; returned status code 404
  [!] Не удалось загрузить 00705.076: Check the url of your file; returned status code 404
  [!] Не удалось загрузить 00706.204: Check the url of your file; returned status code 404
  [!] Не удалось загрузить 00707.339: Check the url of your file; returned status code 404
  [!] Не удалось загрузить 00708.158: Check the url of your file; returned status code 404
  [!] Не удалось загрузить 00709.364: Check the url of your file; returned status code 404


  [!] Не удалось загрузить 00712.013: Check the url of your file; returned status code 404
  [!] Не удалось загрузить 00712.084: Check the url of your file; returned status code 404
  [!] Не удалось загрузить 00801.039: Check the url of your file; returned status code 404
  [!] Не удалось загрузить 00801.479: Check the url of your file; returned status code 404
  [!] Не удалось загрузить 00802.143: Check the url of your file; returned status code 404
  [!] Не удалось загрузить 00803.349: Check the url of your file; returned status code 404


  [!] Не удалось загрузить 00805.148: Check the url of your file; returned status code 404
  [!] Не удалось загрузить 00805.429: Check the url of your file; returned status code 404
  [!] Не удалось загрузить 00806.285: Check the url of your file; returned status code 404
  [!] Не удалось загрузить 00806.289: Check the url of your file; returned status code 404
  [!] Не удалось загрузить 00806.421: Check the url of your file; returned status code 404
  [!] Не удалось загрузить 00809.049: Check the url of your file; returned status code 404
  [!] Не удалось загрузить 00809.127: Check the url of your file; returned status code 404
  [!] Не удалось загрузить 00809.159: Check the url of your file; returned status code 404

Успешно загружено статей: 129
Ошибок загрузки: 21
Всего страниц (Document объектов): 3058


In [6]:
def clean_text(text: str) -> str:
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'-\s+', '', text)
    text = text.strip()
    return text

for doc in all_documents:
    doc.page_content = clean_text(doc.page_content)

all_documents = [doc for doc in all_documents if len(doc.page_content) >= 100]

print(f"Документов после очистки: {len(all_documents)}")
print("\nПример очищенного текста:")
print(all_documents[0].page_content[:300])

Документов после очистки: 3013

Пример очищенного текста:
arXiv:0704.0671v1 [cs.IT] 5 Apr 2007 Learning From Compressed Observations Maxim Raginsky Beckman Institute and University of Illinois 405 N Mathews Ave, Urbana, IL 61801 maxim@uiuc.edu Abstract— The problem of statistical learning is to construct a predictor of a random variable Y as a function of 


## Разбить на чанки

In [7]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=160,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len,
)

chunks = splitter.split_documents(all_documents)

print(f"Всего чанков: {len(chunks)}")
print(f"Среднее число чанков на документ: {len(chunks) / len(all_documents):.1f}")
print(f"Средний размер чанка: {sum(len(c.page_content) for c in chunks) / len(chunks):.0f} символов")
print("\nПример чанка:")
print(repr(chunks[5].page_content[:400]))
print("\nМетаданные чанка:")
print({k: v for k, v in chunks[5].metadata.items() if k != 'pk'})

Всего чанков: 11875
Среднее число чанков на документ: 3.9
Средний размер чанка: 649 символов

Пример чанка:
'. Let Z = (X,Y ) ∼ P be independent of the training sequence Zn. The main quantity of interest is the generalization error of the learner, L( ˆfn,P ) △ = E [ ℓ ( ˆfn(Zn,X ),Y ) ⏐ ⏐ ⏐Zn ] ≡ ∫ Z ℓ( ˆfn(Zn,x ),y )dP (x,y ). The generalization error is a random variable, as it depends on the training sequence Zn. One is chieﬂy interested in the asymptotic probabilistic behavior of the excess lossL( ˆf'

Метаданные чанка:
{'producer': 'dvips + GPL Ghostscript GIT PRERELEASE 9.22', 'creator': 'LaTeX with hyperref package', 'creationdate': '2018-11-04T11:23:26-05:00', 'moddate': '2018-11-04T11:23:26-05:00', 'title': 'Learning from compressed observations', 'subject': '', 'author': '', 'keywords': '', 'source': 'https://arxiv.org/pdf/0704.0671', 'total_pages': 6, 'page': 0, 'page_label': '1', 'arxiv_id': '0704.0671', 'categories': 'cs.IT cs.LG math.IT'}


In [8]:
short_chunks = [c for c in chunks if len(c.page_content) < 50]
print(f"Слишком коротких чанков (< 50 символов): {len(short_chunks)}")

lengths = [len(c.page_content) for c in chunks]
print(f"Мин: {min(lengths)}, Макс: {max(lengths)}, Среднее: {sum(lengths)/len(lengths):.0f}")

Слишком коротких чанков (< 50 символов): 33
Мин: 3, Макс: 800, Среднее: 649


## Создать векторный индекс

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

vectorstore = FAISS.from_documents(chunks, embedding_model)

print(f"Обработано чанков: {vectorstore.index.ntotal}")
print(f"Размерность эмбеддингов: {vectorstore.index.d}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Обработано чанков: 11875
Размерность эмбеддингов: 384


In [10]:
test_query = "transformer attention mechanism"
results = vectorstore.similarity_search(test_query, k=3)

print(f"Тестовый запрос: '{test_query}'\n")
for i, doc in enumerate(results):
    print(f"--- Результат {i+1} ---")
    print(f"Статья: {doc.metadata.get('title', 'N/A')[:80]}")
    print(f"Категории: {doc.metadata.get('categories', 'N/A')}")
    print(f"Текст: {doc.page_content[:200]}...\n")

Тестовый запрос: 'transformer attention mechanism'

--- Результат 1 ---
Статья: Sign Language Tutoring Tool
Категории: cs.LG cs.HC
Текст: . 8 and 9, one can see that this filter p revents fast changes in frames 5 and 6. This filter is designed to work in real-time applications. If used in offline application, it can easily be changed to...

--- Результат 2 ---
Статья: Sign Language Tutoring Tool
Категории: cs.LG cs.HC
Текст: . They consist of a set of 2 high-level (visemes and 6 archetypal emotions) and 66 low-level parameters (depicted as white filled points on Fig. 14). In this project, we only use the lowlevel paramete...

--- Результат 3 ---
Статья: Sign Language Tutoring Tool
Категории: cs.LG cs.HC
Текст: . [10] Benoit A. , Caplier A. "Head Nods Analysis: Interpretation of Non Verbal Com munication Gestures" IEEE, ICIP 2005, Genova, Italy [11] Benoit A. , Caplier A. "Hypovigilence Analysis: Open or Clo...



## Реализовать цепочку

In [11]:
import json
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document

print('Импорты выполнены успешно')


Импорты выполнены успешно


In [12]:
from google.colab import userdata

GEN_MODEL_ID = "llama-3.1-8b-instant"
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
TOP_K = 4

In [13]:
def format_docs(docs):
    return "\n\n".join(
        f"[Paper: {doc.metadata.get('title','N/A')[:60]} | {doc.metadata.get('arxiv_id','N/A')}]\n{doc.page_content}"
        for doc in docs
    )

SYSTEM_PROMPT = """You are a scientific assistant specializing in machine learning research.
Answer the user's question ONLY based on the provided context from ArXiv papers.
If the context does not contain enough information to answer the question, respond:
I don't have enough information in the provided papers to answer this question.
Do not hallucinate. Be concise and precise.
Always mention which paper(s) your answer is based on.

Context from retrieved papers:
{context}"""

PROMPT = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
])

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K}
)

llm = ChatGroq(
    model=GEN_MODEL_ID,
    api_key=GROQ_API_KEY,
    temperature=0.2,
    max_tokens=512,
)

# LCEL RAG chain (langchain 1.x)
rag_chain = (
    RunnableParallel(
        context=retriever | format_docs,
        question=RunnablePassthrough(),
        docs=retriever,
    )
    | RunnableParallel(
        answer=(PROMPT | llm | StrOutputParser()),
        context=lambda x: x["docs"],
        question=lambda x: x["question"],
    )
)

def clip_text(text, threshold=200):
    return f"{text[:threshold]}..." if len(text) > threshold else text

def ask(question: str, verbose: bool = True) -> dict:
    """Задаёт вопрос RAG-системе и возвращает ответ с источниками."""
    resp = rag_chain.invoke(question)

    if verbose:
        print(f"Вопрос:\n{resp['question']}\n")
        print(f"Ответ:\n{resp['answer']}\n")
        print("Использованные чанки:")
        for i, doc in enumerate(resp["context"]):
            print(f"\n  [{i+1}] Статья: {doc.metadata.get('title', 'N/A')[:70]}")
            print(f"       ID: {doc.metadata.get('arxiv_id', 'N/A')}")
            print(f"       Категории: {doc.metadata.get('categories', 'N/A')}")
            print(f"       Текст: {clip_text(doc.page_content)}")
    return resp

print("SMOKE TEST")
_ = ask("What is gradient descent?")


SMOKE TEST
Вопрос:
What is gradient descent?

Ответ:
Gradient descent is often referred to as the update rule in the scenario outlined in the paper "Sparse Online Learning via Truncated Gradient | 0806.4686".

Использованные чанки:

  [1] Статья: Sparse Online Learning via Truncated Gradient
       ID: 0806.4686
       Категории: cs.LG cs.AI
       Текст: . The above method has been widely used in online learning such as [10] and [2]. Moreover, it is argued to be e cient even for solving batch problems where we repeatedly run the online algorithm over ...

  [2] Статья: Model Selection Through Sparse Maximum Likelihood Estimation
       ID: 0707.0704
       Категории: cs.AI cs.LG
       Текст: . First, we begin with some regularization and, as a consequen ce, each penalized regression problem has a unique solution. Second, and more importantly , we update the problem data after each regress...

  [3] Статья: Sparse Online Learning via Truncated Gradient
       ID: 0806.4686
       Кате

In [14]:
print("ТЕСТ: ВОПРОС ВНЕ КОНТЕКСТА")
_ = ask("What is the population of France in 2024?")

ТЕСТ: ВОПРОС ВНЕ КОНТЕКСТА
Вопрос:
What is the population of France in 2024?

Ответ:
I don't have enough information in the provided papers to answer this question.

Использованные чанки:

  [1] Статья: New Estimation Procedures for PLS Path Modelling
       ID: 0802.1002
       Категории: cs.LG
       Текст: Appendices : A Senegalese Departmental Data2 District NHPI PctAgriInc IncActivePers ActivePop Scol Malnutrition DrinkWater Rural Urban PopDensity HouseholdSize Pop0_14 Pop15_60 PopOver60 WIndep WPubli...

  [2] Статья: New Estimation Procedures for PLS Path Modelling
       ID: 0802.1002
       Категории: cs.LG
       Текст: 7.3 48.5 46.9 4.6 68.5 1.0 0.6 29.9 ZIGUINCHOR 40.5 26.4 157 16.5 57.3 4.8 18.5 29.8 70.2 129 8.1 48.1 48.0 3.9 60.7 2.5 15.6 21.2 2 Source : Direction de la Prévision et de la Statistique du Sénégal

  [3] Статья: New Estimation Procedures for PLS Path Modelling
       ID: 0802.1002
       Категории: cs.LG
       Текст: . 0 21030 9.1 46.8 49.9 3.3 46.0 7.2 25

## Протестировать на примерах, оценить качество

### Вопрос 1 — Фактический

In [15]:
print("ВОПРОС 1 (ФАКТИЧЕСКИЙ)")
resp1 = ask("What is the attention mechanism in transformer models and how does it work?")

ВОПРОС 1 (ФАКТИЧЕСКИЙ)
Вопрос:
What is the attention mechanism in transformer models and how does it work?

Ответ:
I don't have enough information in the provided papers to answer this question.

Использованные чанки:

  [1] Статья: Knowledge Technologies
       ID: 0802.3789
       Категории: cs.CY cs.AI cs.LG cs.SE
       Текст: . it uses the knowledge base to alter the contents of the working memory. In addition, most modern systems have 2 further elements: 23

  [2] Статья: A Reactive Tabu Search Algorithm for Stimuli Generation in   Psycholin
       ID: 0712.0451
       Категории: cs.AI cs.CC cs.DM cs.LG
       Текст: . Two major tools in this research area are computational models and laboratory experiments in which language features are manipulated. Computational models try to simulate how language information is...

  [3] Статья: Sign Language Tutoring Tool
       ID: 0802.2428
       Категории: cs.LG cs.HC
       Текст: . 8 and 9, one can see that this filter p revents fast ch

Этот вопрос направлен на извлечение конкретного технического определения. Ожидаем, что система найдёт чанки из статей, описывающих механизм внимания (attention mechanism).

Качество retrieval: Механизм внимания — базовая концепция в области cs.LG, поэтому вероятность найти релевантные чанки высока. FAISS должен корректно сопоставить вопрос с фрагментами, где встречается термин "attention".

Качество ответа: Ожидаем точное, технически грамотное определение. Если чанки содержат оригинальный текст из статей по трансформерам — ответ будет хорошим. Если попались только упоминания без объяснений — ответ может быть поверхностным.

Что можно улучшить: Использовать более тематически специализированный корпус (например, только статьи про трансформеры).

### Вопрос 2 — Обобщающий

In [16]:
print("ВОПРОС 2 (ОБОБЩАЮЩИЙ)")
resp2 = ask("What are the main challenges and limitations of training large language models according to the papers?")

ВОПРОС 2 (ОБОБЩАЮЩИЙ)
Вопрос:
What are the main challenges and limitations of training large language models according to the papers?

Ответ:
Based on the papers, the main challenges and limitations of training large language models are:

1. **Convergence to a global optimum**: There is no guarantee that convergence to a global optimum for any particular cost function will occur, even with existing NMF algorithms (Paper: Positive factor networks: A graphical framework for modeling | 0807.4198).
2. **Limited performance on hard real-world problems**: It is still unknown how well models using the proposed framework (PFN) will perform on hard real-world problems such as speech recognition, language modeling, and music transcription (Paper: Positive factor networks: A graphical framework for modeling | 0807.4198).
3. **Limited data**: The model in Paper: Utilisation des grammaires probabilistes dans les t\^aches d | 0806.1156 was trained on a limited corpus, which may not be representative

Обобщающий вопрос требует синтеза информации из разных источников.

Качество retrieval: Семантический поиск должен найти чанки, где упоминаются проблемы обучения (вычислительная сложность, overfitting, hallucinations и т.д.). Возможно, retrieved чанки будут из разных статей — это хорошо для обобщающего вопроса.

Качество ответа: RAG-система хорошо справляется с обобщением, если в TOP_K чанках перечислены конкретные проблемы. Однако модель может «смешивать» информацию из разных контекстов — важно проверить, ссылается ли ответ на конкретные источники.

Ограничение: 4 чанка (TOP_K=4) могут не покрыть все аспекты темы. Для обобщающих вопросов TOP_K можно увеличить до 6–8.


### Вопрос 3 — Уточняющий

In [17]:
print("ВОПРОС 3 (УТОЧНЯЮЩИЙ)")
resp3 = ask("What specific hyperparameters and training tricks are recommended for fine-tuning BERT-like models on classification tasks?")

ВОПРОС 3 (УТОЧНЯЮЩИЙ)
Вопрос:
What specific hyperparameters and training tricks are recommended for fine-tuning BERT-like models on classification tasks?

Ответ:
I don't have enough information in the provided papers to answer this question.

Использованные чанки:

  [1] Статья: Combining Expert Advice Efficiently
       ID: 0802.2015
       Категории: cs.LG cs.DS cs.IT math.IT
       Текст: References [1] O. Bousquet. A note on parameter tuning for on-line shift ing algorithms. Technical report, Max Planck Institute for Biologi cal Cybernetics, 2003. [2] O. Bousquet and M. K. Warmuth. Tr...

  [2] Статья: A Uniform Approach to Analogies, Synonyms, Antonyms, and Associations
       ID: 0809.0124
       Категории: cs.CL cs.IR cs.LG
       Текст: . We then normalize all of the phrases that are found, by using morpha to remove sufﬁxes. The template we use here is similar to Turney (2006), but we have added extra context words before the X and a...

  [3] Статья: Clustered Multi-Task Learn

Уточняющий вопрос фокусируется на конкретных гиперпараметрах — информации, которая может присутствовать в статьях, но только в экспериментальных разделах.

Качество retrieval: Вероятность найти точный ответ ниже, чем для фактического вопроса. Semantic search ищет по смыслу, а конкретные значения гиперпараметров (learning rate=2e-5 и т.д.) часто «утоплены» в таблицах или footnote-секциях PDF, которые плохо парсятся.

Качество ответа: Если чанки содержат экспериментальные секции — ответ будет конкретным. Если нет — модель либо честно ответит «не знаю» (хорошо!), либо даст общие рекомендации из своих параметров (плохо, но ожидаемо).

Вывод: Уточняющие вопросы выявляют границы RAG: система работает хорошо на «концептуальном» уровне, но хуже на «цифровом».

### Вопрос 4 — Сравнительный (бонус)
Сравнительный вопрос требует сопоставления подходов из разных статей.

In [18]:
print("ВОПРОС 4 (СРАВНИТЕЛЬНЫЙ)")
resp4 = ask("How do recurrent neural networks (RNNs) compare to transformers for sequence modeling tasks?")

ВОПРОС 4 (СРАВНИТЕЛЬНЫЙ)
Вопрос:
How do recurrent neural networks (RNNs) compare to transformers for sequence modeling tasks?

Ответ:
I don't have enough information in the provided papers to answer this question.

Использованные чанки:

  [1] Статья: The structure of verbal sequences analyzed with unsupervised learning 
       ID: 0710.2446
       Категории: cs.CL cs.AI cs.LG
       Текст: . From this coding, we took the Hidden Markov Models (HMM) to model the dynamics of the sequences of data (here, the verbs within a sentence). The HMM (Rabiner and Juang, 1986) are the best approach t...

  [2] Статья: Positive factor networks: A graphical framework for modeling   non-neg
       ID: 0807.4198
       Категории: cs.LG
       Текст: . As with existing NMF algorithms, however, there is no guarantee that convergence to a global optimum for any particular cost function will occur. We did observe good convergence properties in our em...

  [3] Статья: Positive factor networks: A graphical 

Качество retrieval: Оба термина (RNN, transformer) часто встречаются вместе в работах по sequence modeling — retrieval должен сработать хорошо.

Качество ответа: Сравнение требует одновременного наличия информации о двух подходах в retrieved чанках. Если статьи посвящены только трансформерам — ответ будет однобоким.

Вывод: RAG справляется со сравнительными вопросами лучше, если корпус включает статьи обоих направлений.

Итоговый вывод

**Что получилось хорошо**

Загрузка и обработка данных прошла нормально: PDF-файлы статей скачались с arxiv.org, лишние символы и переносы почистились, метаданные сохранились. Разбиение на чанки работает нормально: фрагменты получаются читаемыми, без оборванных предложений. Поиск по индексу на простых вопросах про трансформеры и механизм внимания находит нужные куски текста. Модель при этом отвечает «не знаю», когда вопрос выходит за пределы загруженных статей.

**Где система работает хуже**

Хуже всего система справляется с конкретными числовыми вопросами, например, про значения learning rate или batch size. На широкие обобщающие вопросы система тоже отвечает неполно: четырёх найденных чанков не хватает, чтобы покрыть тему со всех сторон.

**Что можно улучшить**

Самый простой шаг — увеличить корпус, это сразу повысит покрытие. Из технических улучшений: добавить поиск по ключевым словам (BM25) в дополнение к семантическому — это помогает на конкретных фактических вопросах. Ещё можно добавить шаг переранжирования результатов, чтобы из 20 найденных чанков отбирались действительно самые релевантные, а не просто похожие по смыслу.